# LLM Agents Self Practice Work book

## What is Agent?

#### A loop of LLM tools for a desired goal.

In [34]:
from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)

True

In [46]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [47]:
import os
gemini = OpenAI(api_key=os.getenv('GEMINI_API_KEY'), base_url="https://generativelanguage.googleapis.com/v1beta/openai/")

In [37]:
# Lists for Trackers

todos = []
completed = []

In [48]:
def get_todo_report() -> str:
    result = ""
    for index, todo in enumerate(todos):
        if completed[index]:
            result += f"Todo #{index + 1}: [green][strike]{todo}[/strike][/green]\n"
        else:
            result += f"Todo #{index + 1}: {todo}\n"    
    show(result)
    return result

In [49]:
get_todo_report()

Todo #1: Listening to podcast
Todo #2: Learning Agents
Todo #3: Starting to work on Deliver Mobility tasks
Todo #4: Leviticus 6 & 7

'Todo #1: [green][strike]Listening to podcast[/strike][/green]\nTodo #2: Learning Agents\nTodo #3: Starting to work on Deliver Mobility tasks\nTodo #4: Leviticus 6 & 7\n'

In [40]:
def create_todos(descriptions: list[str]) -> str:
    todos.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_todo_report()

In [42]:
def mark_complete(index: int, completion_notes: str) -> str: 
    if 1 <= index <= len(todos):
        completed[index - 1] = True
    else:
        return "No todo at this index."
    Console().print(completion_notes)
    return get_todo_report()

In [43]:
todos, completed = [], []

create_todos(["Listening to podcast","Learning Agents", "Starting to work on Deliver Mobility tasks", "Leviticus 6 & 7"])

Todo #1: Listening to podcast
Todo #2: Learning Agents
Todo #3: Starting to work on Deliver Mobility tasks
Todo #4: Leviticus 6 & 7



'Todo #1: Listening to podcast\nTodo #2: Learning Agents\nTodo #3: Starting to work on Deliver Mobility tasks\nTodo #4: Leviticus 6 & 7\n'

In [50]:
mark_complete(1, "Listening to Dejaf Podcast")

Listening to Dejaf Podcast

Todo #1: Listening to podcast
Todo #2: Learning Agents
Todo #3: Starting to work on Deliver Mobility tasks
Todo #4: Leviticus 6 & 7

'Todo #1: [green][strike]Listening to podcast[/strike][/green]\nTodo #2: Learning Agents\nTodo #3: Starting to work on Deliver Mobility tasks\nTodo #4: Leviticus 6 & 7\n'

In [51]:

create_todos_json = {
    "name": "create_todos",
    "description": "Add new todos from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [52]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the todo at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the todo to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the todo in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [53]:
tools = [{"type": "function", "function": create_todos_json},
        {"type": "function", "function": mark_complete_json}]

In [54]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool", "content": json.dumps(result), "tool_call_id":tool_call.id})
    return results

In [66]:
def loop(messages):
    done = False
    while not done:
        response = gemini.chat.completions.create(model='gemini-2.5-flash', messages=messages, tools=tools, reasoning_effort="none")
        finish_reason = response.choices[0].finish_reason
        if finish_reason =="tool_calls":
            message = response.choices[0].message
            tool_calls = message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(message)
            messages.extend(results)
        else:
            done = True
    show(response.choices[0].message.content)    

In [68]:
system_message = """
You are given a problem to solve, by using your todo tools to plan a list of steps, then carrying out each step in turn.
Now use the todo list tools, create a plan, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """
Hey, I'm working on some physics problems. Can you calculate the weight (in Newtons) of a 15kg object on Earth (use g = 9.8 m/s²)? Once you have that value, add a new todo that says: 'Buy a scale that reads up to [calculated weight] N'. 

After that, check off whatever task is currently at index 2 on my list, and set the completion notes to: 'Calculated using F = ma'.
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [69]:
todos, completed = [], []

loop(messages)

Todo #1: Buy a scale that reads up to 147.0 N

The weight of a 15kg object on Earth (using g = 9.8 m/s²) is 147.0 Newtons. I have added a todo to "Buy a scale 
that reads up to 147.0 N". I attempted to check off the task at index 2, but there was no todo at that index.